# ADSP 31013 IP01 Big Data and Cloud Computing

## Group Project – NYC Yellow Taxi Trips (2024/01/01 - 2025/10/15)

### Meteostat Developers API Weather Fetching Pipeline

This notebook implements the weather-data fetching pipeline used in our Big Data Group Project.
Its goal is to download and export hourly New York weather observations by Borough and Zone, which will later be joined to NYC Yellow Taxi trip records to support feature engineering (e.g., temperature, precipitation, wind speed, atmospheric pressure at pickup and dropoff times).

This notebook produces the pickup-time and dropoff-time weather datasets that are eventually merged into the main taxi dataset in our EDA and ML notebooks.

Data source: 
**New York City Hourly Weather**, provided by:
* **Meteostat Developers** ([https://dev.meteostat.net/api/point/hourly.html#endpoint](https://dev.meteostat.net/api/point/hourly.html#endpoint))

**Author(s):** Zihao Huang(zihaoh3@uchicago.edu), Ray Gong(ruigong1307@uchicago.edu), Leo Liu(jliu888@uchicago.edu), Wade Chen(wangdi@uchicago.edu), Tom Chen(cychen1@uchicago.edu)

#### 0. Imports and `meteostat` package download

In [1]:
# !pip install meteostat

In [ ]:
from meteostat import Stations, Hourly
from datetime import datetime
import pandas as pd

#### 1. Set up Station Information

**Define the weather stations relevant to the New York City area**

This section specifies the Latitude/longitude of each Borough-Zone. And these stations will be queried for hourly weather observations. They are selected because they are representative of NYC climate patterns and suitable for merging with taxi trip timestamps.

In [4]:
df = pd.read_csv("gs://msca-bdp-student-gcs/group_8_project/datasets/taxi_zone.csv")
df.head()

,LocationID,Borough,Zone,service_zone,latitude,longitude
0,1,NaN,NaN,NaN,NaN,NaN
1,2,Queens,Jamaica Bay,Boro Zone,40.615625,-73.831063
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone,40.862750,-73.847207
3,4,Manhattan,Alphabet City,Yellow Zone,40.722369,-73.976936
4,5,Staten Island,Arden Heights,Boro Zone,40.551695,-74.188749


In [5]:
# Set time period
start = datetime(2024, 1, 1)
end = datetime(2025, 10, 31, 23, 59)
out_path = "gs://msca-bdp-student-gcs/group_8_project/datasets/raw/weather/"

# iterate the `df` by rows
for index, row in df.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    if pd.isna(lat) or pd.isna(lon):
        print(f"Skipping LocationID: {row['LocationID']}, Borough: {row['Borough']}, Zone: {row['Zone']} due to missing lat/lon")
        continue

    print(f"Fetching weather data for LocationID: {row['LocationID']}, Borough: {row['Borough']}, Zone: {row['Zone']} at lat: {lat}, lon: {lon}")
    try:
        # Get nearby weather stations
        stations = Stations()
        stations = stations.nearby(lat, lon)
        station = stations.fetch(1)
        
        # Get hourly data
        data = Hourly(station.index[0], start, end)
        data = data.fetch()

    except Exception as e:
        print(f"Error fetching weather station for LocationID: {row['LocationID']}, Borough: {row['Borough']}, Zone: {row['Zone']}: {e}")
        continue


    # Add location ID, Borough, Zone columns
    data['LocationID'] = row['LocationID']
    data['Borough'] = row['Borough']
    data['Zone'] = row['Zone']

    # move LocationID, Borough, Zone columns to the front
    cols = data.columns.tolist()
    cols = cols[-3:] + cols[:-3]
    data = data[cols]

    # Save to CSV
    # Remove any '/' in Borough or Zone to avoid S3 path issues
    row['Borough'] = row['Borough'].replace('/', '_')
    row['Zone'] = row['Zone'].replace('/', '_')
    filename = f"weather_data_zone_{row['LocationID']}_{row['Borough']}_{row['Zone']}_2024.csv"
    data.to_csv(out_path + filename)

Skipping LocationID: 1, Borough: nan, Zone: nan due to missing lat/lon
Fetching weather data for LocationID: 2, Borough: Queens, Zone: Jamaica Bay at lat: 40.6156247016788, lon: -73.8310628287166
Fetching weather data for LocationID: 3, Borough: Bronx, Zone: Allerton/Pelham Gardens at lat: 40.8627500836362, lon: -73.8472066667145
Fetching weather data for LocationID: 4, Borough: Manhattan, Zone: Alphabet City at lat: 40.7223693366811, lon: -73.9769358706712
Fetching weather data for LocationID: 5, Borough: Staten Island, Zone: Arden Heights at lat: 40.5516947609216, lon: -74.1887490985966
Fetching weather data for LocationID: 6, Borough: Staten Island, Zone: Arrochar/Fort Wadsworth at lat: 40.5992433491202, lon: -74.0718710343035
Fetching weather data for LocationID: 7, Borough: Queens, Zone: Astoria at lat: 40.7600181060481, lon: -73.9195812887703
Fetching weather data for LocationID: 8, Borough: Queens, Zone: Astoria Park at lat: 40.7770427209948, lon: -73.9229779633358
Fetching weat

#### 2. **Conclusion**

In this notebook, we successfully implemented a complete data-fetching pipeline that retrieves and export hourly weather observations for the New York City area. These weather measurements serve as an important external dataset that will later be joined to NYC Yellow Taxi pickup and dropoff timestamps in our feature-engineering workflow.

**Columns:**

* **time** — The timestamp of the weather observation (UTC or local-converted depending on the station’s reporting format).
* **LocationID** — The mapped NYC Taxi Zone ID associated with the weather station.
* **Borough** — The corresponding borough for the weather station.
* **Zone** — The NYC TLC taxi zone name matching the station region.
* **temp** — Air temperature (°C).
* **dwpt** — Dew point temperature (°C).
* **rhum** — Relative humidity (%).
* **prcp** — Precipitation amount (mm).
* **snow** — Snowfall amount.
* **wdir** — Wind direction (degrees).
* **wspd** — Wind speed (m/s).
* **wpgt** — Peak wind gust (m/s).
* **pres** — Atmospheric pressure (hPa).
* **tsun** — Sunshine duration.
* **coco** — Weather condition code indicating cloudiness or other meteorological phenomena.